In [0]:
dbutils.widgets.text("job_run_id", "")

job_run_id = dbutils.widgets.get("job_run_id")

print("Job Run ID:", job_run_id)

In [0]:
import requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

host = ctx.apiUrl().get()
token = ctx.apiToken().get()

response = requests.get(
    f"{host}/api/2.2/jobs/runs/get",
    headers={"Authorization": f"Bearer {token}"},
    params={"run_id": job_run_id},
    timeout=30
)

response.raise_for_status()
run_info = response.json()

for task in run_info.get("tasks", []):
    print(
        task["task_key"],
        "|",
        task.get("state", {}).get("result_state", "NOT_FINISHED")
    )

In [0]:

from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DateType, TimestampType, LongType
)
from delta.tables import DeltaTable

# 1. Define upstream tasks
upstream_tasks = [
    "M3_FINAL_01_Bronze_to_Silver",
    "M3_FINAL_02_Update_DQ_Log",
    "M5_dbt_Gold_Build"
]

# 2. Collect task results
task_results = {
    task["task_key"]: task.get("state", {}).get("result_state")
    for task in run_info.get("tasks", [])
}

# 3. Determine pipeline status
pipeline_status = (
    "SUCCESS"
    if all(task_results.get(t) == "SUCCESS" for t in upstream_tasks)
    else "FAILED"
)

failed_tasks = [
    f"{t}: {task_results.get(t) or 'NOT_AVAILABLE'}"
    for t in upstream_tasks
    if task_results.get(t) != "SUCCESS"
]

# 4. Capture timestamps
start_time = datetime.fromtimestamp(
    run_info["start_time"] / 1000,
    tz=timezone.utc
).replace(tzinfo=None)

# Use the latest completed upstream task end time.
# The overall job may still be running while this audit task executes.
upstream_end_times = [
    task["end_time"]
    for task in run_info.get("tasks", [])
    if task["task_key"] in upstream_tasks
    and task.get("end_time", 0) > 0
]

end_time = datetime.fromtimestamp(
    max(upstream_end_times) / 1000,
    tz=timezone.utc
).replace(tzinfo=None) if upstream_end_times else datetime.now(timezone.utc).replace(tzinfo=None)

# 5. Define explicit schema
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), True),
    StructField("run_date", DateType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("error_message", StringType(), True)
])

# 6. Create audit DataFrame
audit_df = spark.createDataFrame(
    [(
        str(job_run_id),
        "JOB_M5_Silver_Processing",
        start_time.date(),
        start_time,
        end_time,
        pipeline_status,
        None,
        "; ".join(failed_tasks) if failed_tasks else None
    )],
    schema=audit_schema
)

# 7. MERGE to prevent duplicate run IDs
(
    DeltaTable.forName(spark, "atliq.silver.pipeline_run_audit")
    .alias("t")
    .merge(
        audit_df.alias("s"),
        "t.run_id = s.run_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Audit recorded:", job_run_id, pipeline_status)


In [0]:
# Capture current table row counts
table_counts = {}

tables = [
    "atliq.silver.customers",
    "atliq.silver.products",
    "atliq.silver.supplier_price_list",
    "atliq.silver.marketing_spend",
    "atliq.silver.web_events",
    "atliq.silver.orders",
    "atliq.silver.order_items",
    "atliq.silver.payments",
]

# Add the 5 Gold tables using their exact names.

for table in tables:
    table_counts[table] = spark.table(table).count()

rows_processed = sum(table_counts.values())

for table, count in table_counts.items():
    print(f"{table}: {count}")

print("Total rows:", rows_processed)

In [0]:
gold_tables = [
    "atliq.gold.dim_customer",
    "atliq.gold.dim_date",
    "atliq.gold.dim_product",
    "atliq.gold.fact_sales",
    "atliq.gold.fact_web_sessions"
]

gold_counts = {}

for table in gold_tables:
    gold_counts[table] = spark.table(table).count()

for table, count in gold_counts.items():
    print(f"{table}: {count}")

print("Total Gold rows:", sum(gold_counts.values()))

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Combine Silver and Gold counts
all_counts = {**table_counts, **gold_counts}

count_rows = [
    (str(job_run_id), table, count)
    for table, count in all_counts.items()
]

counts_df = (
    spark.createDataFrame(
        count_rows,
        ["run_id", "table_name", "row_count"]
    )
    .withColumn("recorded_at", F.current_timestamp())
)

# Idempotent write: one record per run and table
(
    DeltaTable.forName(
        spark, "atliq.silver.pipeline_table_counts"
    )
    .alias("t")
    .merge(
        counts_df.alias("s"),
        "t.run_id = s.run_id AND t.table_name = s.table_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Saved {len(count_rows)} table counts for run {job_run_id}")